In [1]:
import os

In [2]:
%pwd

'e:\\repo\\CICD\\CI_CD3_DL\\research'

In [3]:
os.chdir('../')

In [5]:
%pwd

'e:\\repo\\CICD\\CI_CD3_DL'

# Entity

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

# Configuration Manager

In [8]:
from src.projectDL_1.constants import *
from src.projectDL_1.utils.common import read_yaml, create_directories

In [9]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config=self.config.data_transformation

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=config.tokenizer_name
        )
        return data_transformation_config




# Components

In [10]:
import os
from src.projectDL_1 import logger 
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

ModuleNotFoundError: No module named 'transformers'

In [11]:
pip install transformers datasets

     --------------------------------------- 10.8/10.8 MB 29.7 MB/s eta 0:00:00
     ---------------------------------------- 529.0/529.0 kB ? eta 0:00:00
     ------------------------------------- 671.5/671.5 kB 21.3 MB/s eta 0:00:00
     ------------------------------------- 278.4/278.4 kB 16.8 MB/s eta 0:00:00
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
     -------------------------------------- 122.4/122.4 kB 7.5 MB/s eta 0:00:00
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)
  Using cached dill-0.4.1-py3-none-any.whl (120 kB)
     ---------------------------------------- 144.5/144.5 kB ? eta 0:00:00
  Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
     ---------------------------------------- 4.0/4.0 MB 42.2 MB/s eta 0:00:00
     ---------------------------------------- 58.4/58.4 kB ? eta 0:00:00
  Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
     ------------------------------------- 310.7/310.7 kB 18.8 MB/s eta 0:

ERROR: Exception:
Traceback (most recent call last):
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\cli\base_command.py", line 160, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\cli\req_command.py", line 247, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\commands\install.py", line 494, in run
    installed = install_given_reqs(
                ^^^^^^^^^^^^^^^^^^^
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\req\__init__.py", line 73, in install_given_reqs
    requirement.install(
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\req\req_install.py", line 792, in install
    install_wheel(
  File "e:\repo\CICD\CI_CD3_DL\env\Lib\site-packages\pip\_internal\operations\install\wheel.py", line 729, in install_wheel
    _install_

In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
    
    def convert_examples_to_features(self, example):
        input_text = example['dialogue']
        target_text = example['summary']

        input_encoding = self.tokenizer(
            input_text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target_encoding = self.tokenizer(
            target_text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        datset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        datset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, 'samsum_dataset_pt'))




In [ ]:
try: 
    config = ConfiguartionManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e